Problem Statement : Implement persistance concept
For the given topic, generate a joke. then generate explaination for that joke.

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver 
# Used as an in-memory checkpointer to save and restore the graph's state in RAM during execution (temporary persistence).

d:\Working with LLM\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm = ChatMistralAI()

In [4]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explaination : str

In [5]:
def generate_joke(state:JokeState):
    prompt = f"Generate a joke on topic : {state['topic']}"
    response = llm.invoke(prompt).content
    return {'joke':response}

In [6]:
def generate_explaination(state: JokeState):
    prompt = f'write an explaination for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {'explaination': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

# Create an in-memory checkpointer to save the workflow state (checkpoints).
checkpointer = InMemorySaver()

# Compile the graph and attach the checkpointer to enable persistence.
workflow = graph.compile(checkpointer=checkpointer)

In [9]:
# Configuration dictionary passed to the workflow.
# 'thread_id' uniquely identifies a conversation/workflow thread for persistence.
config1 = {"configurable": {"thread_id": "1"}}

# Execute the workflow with the given input and configuration.
# LangGraph saves and retrieves the workflow state using thread_id "1".
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'What kind of pizza do ghosts eat? *Boo*-beroni!',
 'explaination': 'This joke is a playful pun that combines two things: ghosts and pizza.\n\n1. **Ghosts and "Boo"**: Ghosts are often associated with the word "boo" because that\'s a common sound people make to scare others, mimicking a ghost\'s "voice."\n\n2. **Pizza and "Booberoni"**: The joke replaces the word "pepperoni" (a common pizza topping) with "booberoni," which sounds like "boo" + "pepperoni."\n\nSo, when someone asks, "What kind of pizza do ghosts eat?" the punchline "*Boo*-beroni!" tricks the listener into expecting a spooky-themed pizza but delivers a simple pun instead.\n\nThe humor comes from the unexpected twist of the pun, making it a lighthearted and silly joke.'}

In [10]:
# Retrieve the latest saved state (checkpoint) of the workflow
# associated with the specified thread_id.
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'What kind of pizza do ghosts eat? *Boo*-beroni!', 'explaination': 'This joke is a playful pun that combines two things: ghosts and pizza.\n\n1. **Ghosts and "Boo"**: Ghosts are often associated with the word "boo" because that\'s a common sound people make to scare others, mimicking a ghost\'s "voice."\n\n2. **Pizza and "Booberoni"**: The joke replaces the word "pepperoni" (a common pizza topping) with "booberoni," which sounds like "boo" + "pepperoni."\n\nSo, when someone asks, "What kind of pizza do ghosts eat?" the punchline "*Boo*-beroni!" tricks the listener into expecting a spooky-themed pizza but delivers a simple pun instead.\n\nThe humor comes from the unexpected twist of the pun, making it a lighthearted and silly joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18d81b-4b88-6b14-8002-09d7f3cbecb3'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-0

In [11]:
# Retrieve the complete checkpoint history of the workflow
# for the specified thread_id and convert it into a list for easy viewing.
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'What kind of pizza do ghosts eat? *Boo*-beroni!', 'explaination': 'This joke is a playful pun that combines two things: ghosts and pizza.\n\n1. **Ghosts and "Boo"**: Ghosts are often associated with the word "boo" because that\'s a common sound people make to scare others, mimicking a ghost\'s "voice."\n\n2. **Pizza and "Booberoni"**: The joke replaces the word "pepperoni" (a common pizza topping) with "booberoni," which sounds like "boo" + "pepperoni."\n\nSo, when someone asks, "What kind of pizza do ghosts eat?" the punchline "*Boo*-beroni!" tricks the listener into expecting a spooky-themed pizza but delivers a simple pun instead.\n\nThe humor comes from the unexpected twist of the pun, making it a lighthearted and silly joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18d81b-4b88-6b14-8002-09d7f3cbecb3'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-

Adding one more thread/chat version

In [12]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to the gym?\n\nBecause it wanted to get a little *al dente*!',
 'explaination': 'The joke plays on two meanings of the phrase "a little *al dente*":\n\n1. **Literal Meaning**: "*Al dente*" is an Italian term used to describe pasta that is cooked to be firm when bitten, rather than soft. So, the phrase "a little *al dente*" could mean the pasta wants to become firmer or more structurally sound.\n\n2. **Pun Meaning**: The phrase "get a little" is often used in the context of exercising or working out (e.g., "get a little stronger," "get a little fitter"). By combining this with "*al dente*," the joke implies that the pasta is going to the gym to become firmer or more resilient, just like when pasta is cooked to the ideal texture.\n\nThe humor comes from the unexpected twist of applying a cooking term to a fitness context, making it a playful and clever pun.'}

Time Travel

In [14]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f18d81b-3102-646b-8000-2d48368cefbc"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f18d81b-3102-646b-8000-2d48368cefbc'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-01T08:19:21.967319+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18d81b-307c-6406-bfff-c5903648561b'}}, tasks=(PregelTask(id='95378c5b-f8ad-f80c-d72e-3d63ae88dba7', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'What kind of pizza do ghosts eat? *Boo*-beroni!'}),), interrupts=())

In [15]:
workflow.invoke(None,{"configurable": {"thread_id": "1", "checkpoint_id": "1f18d81b-3102-646b-8000-2d48368cefbc"}})

{'topic': 'pizza',
 'joke': "What do you call cheese that isn't yours? Nacho cheese!",
 'explaination': 'The joke plays on the phrase *"nacho cheese"* by misinterpreting it as *"not your cheese."* The humor comes from the pun, where the listener expects a straightforward answer (like "someone else\'s cheese"), but instead gets a playful twist that sounds like a silly name for cheese. The word *"nacho"* sounds like *"not your,"* making it a clever and simple pun that relies on wordplay rather than a complex setup.'}

Update State

In [16]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f18d81b-3102-646b-8000-2d48368cefbc", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f18d831-e7a4-6314-8001-71e9922462a6'}}

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18d831-e7a4-6314-8001-71e9922462a6'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-08-01T08:29:31.675726+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18d81b-3102-646b-8000-2d48368cefbc'}}, tasks=(PregelTask(id='7216efd0-880a-c90f-58a8-559018991758', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "What do you call cheese that isn't yours? Nacho cheese!", 'explaination': 'The joke plays on the phrase *"nacho cheese"* by misinterpreting it as *"not your cheese."* The humor comes from the pun, where the listener expects a straightforward answer (like "someone else\'s cheese"), but instead gets a playful twist that sounds lik

In [18]:
workflow.invoke(None,{"configurable": {"thread_id": "1", "checkpoint_id": "1f18d831-e7a4-6314-8001-71e9922462a6"}})

{'topic': 'samosa',
 'joke': "Here's a samosa joke for you:\n\n**Why did the samosa go to therapy?**\nBecause it had too many *layers* of emotional baggage—and now it’s feeling *stuffed*! 🥟💔",
 'explaination': 'This joke plays on multiple layers of meaning, both literal and figurative, to create humor. Here\'s a breakdown of how it works:\n\n1. **Literal Meaning of "Samosa"**\n   A samosa is a popular fried or baked pastry with a flaky, layered dough filled with spiced potatoes, meat, or other ingredients. The joke starts with this familiar image.\n\n2. **Emotional Baggage (Figurative Meaning)**\n   "Emotional baggage" is a common idiom referring to past emotional trauma or unresolved issues that someone carries around, causing them stress. The samosa is humorously anthropomorphized—given human-like qualities—by suggesting it *has* emotional baggage.\n\n3. **Layers (Double Meaning)**\n   - *Literal*: A samosa is made of layers (of dough, filling, etc.).\n   - *Figurative*: "Layers" can